# 02 — Split Generator (Run only after Gate 1 PASS)
Generates one stratified 70/15/15 split.csv with seed 42. Do not run before Dataset Audit approval.

> **FROZEN on 2026-10-02 (DECISION_LOG D007) — do not re-run.**
> `split.csv` SHA256 `043e68e8028e402f2c2d04cdfa48e44b54c9ac6099981387f2859a0724aa4b1b`,
> `class_mapping.csv` SHA256 `a99922d406b2a308ff6e85e0015dc675ad24be07f7e481ff4f0a1ed65a8aed8d`.
> The split is reproducible as a frozen file, not from code: `train_test_split` assigns rows by their position in `image_metadata.csv`, whose order comes from the file listing. A guard cell below refuses to overwrite the frozen files. Load the split through `src/dataset.py`, which verifies both hashes.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42

AUDIT_OUT = Path(
    "/content/drive/MyDrive/Deep Learning - RealWaste/04_Results/dataset_audit"
)

SPLIT_DIR = Path(
    "/content/drive/MyDrive/Deep Learning - RealWaste/02_Splits"
)

SPLIT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(AUDIT_OUT / "image_metadata.csv")

print("Metadata rows:", len(df))
print("Classes:", df["class_name"].nunique())

assert len(df) == 4752, f"Unexpected number of images: {len(df)}"
assert df["class_name"].nunique() == 9

Metadata rows: 4752
Classes: 9


In [4]:
from sklearn.model_selection import train_test_split

# First split: 70% train, 30% temp
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["class_name"],
    random_state=SEED
)

# Second split: split temp equally into 15% val, 15% test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["class_name"],
    random_state=SEED
)

train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df["split"] = "train"
val_df["split"] = "val"
test_df["split"] = "test"

split_df = pd.concat(
    [train_df, val_df, test_df],
    ignore_index=True
)

# Keep deterministic ordering
split_df = split_df.sort_values(
    ["split", "class_name", "filepath"]
).reset_index(drop=True)

print("Total:", len(split_df))
print(split_df["split"].value_counts())

print("\nRatios:")
print(
    (split_df["split"].value_counts() / len(split_df))
    .round(4)
)

Total: 4752
split
train    3326
test      713
val       713
Name: count, dtype: int64

Ratios:
split
train    0.6999
test     0.1500
val      0.1500
Name: count, dtype: float64


In [5]:
distribution = pd.crosstab(
    split_df["class_name"],
    split_df["split"]
)

distribution["total"] = distribution.sum(axis=1)

distribution["train_pct"] = (
    distribution["train"] / distribution["total"] * 100
).round(2)

distribution["val_pct"] = (
    distribution["val"] / distribution["total"] * 100
).round(2)

distribution["test_pct"] = (
    distribution["test"] / distribution["total"] * 100
).round(2)

print(distribution)

split                test  train  val  total  train_pct  val_pct  test_pct
class_name                                                                
Cardboard              69    323   69    461      70.07    14.97     14.97
Food Organics          62    288   61    411      70.07    14.84     15.09
Glass                  63    294   63    420      70.00    15.00     15.00
Metal                 118    553  119    790      70.00    15.06     14.94
Miscellaneous Trash    75    346   74    495      69.90    14.95     15.15
Paper                  75    350   75    500      70.00    15.00     15.00
Plastic               138    645  138    921      70.03    14.98     14.98
Textile Trash          48    222   48    318      69.81    15.09     15.09
Vegetation             65    305   66    436      69.95    15.14     14.91


In [6]:
assert len(split_df) == len(df)
assert split_df["filepath"].is_unique, "Duplicate filepath detected!"

train_paths = set(train_df["filepath"])
val_paths = set(val_df["filepath"])
test_paths = set(test_df["filepath"])

assert train_paths.isdisjoint(val_paths)
assert train_paths.isdisjoint(test_paths)
assert val_paths.isdisjoint(test_paths)

print("File-level split leakage: NONE ✅")

File-level split leakage: NONE ✅


In [ ]:
# FROZEN-SPLIT GUARD (added 2026-10-02, DECISION_LOG D007).
# Re-running the cells above can produce a DIFFERENT split with the same seed, so the cells
# below must never overwrite the frozen files. Set to True only after an approved
# DECISION_LOG entry that supersedes D007.
ALLOW_SPLIT_REGENERATION = False

frozen_files = [SPLIT_DIR / name for name in ("split.csv", "class_mapping.csv", "split_manifest.txt")]
existing_frozen = [str(p) for p in frozen_files if p.exists()]

if existing_frozen and not ALLOW_SPLIT_REGENERATION:
    raise RuntimeError(
        "Frozen split files already exist; refusing to overwrite them:\n  "
        + "\n  ".join(existing_frozen)
        + "\nLoad the frozen split through src/dataset.py (SHA256-verified) instead."
    )
print("No frozen split files found (or regeneration approved); writing split files.")

In [7]:
split_path = SPLIT_DIR / "split.csv"

split_df.to_csv(
    split_path,
    index=False
)

print("Saved:", split_path)
print("Exists:", split_path.exists())

Saved: /content/drive/MyDrive/Deep Learning - RealWaste/02_Splits/split.csv
Exists: True


In [8]:
class_names = sorted(split_df["class_name"].unique())

class_to_idx = {
    class_name: idx
    for idx, class_name in enumerate(class_names)
}

mapping_df = pd.DataFrame({
    "class_name": class_names,
    "class_index": range(len(class_names))
})

mapping_path = SPLIT_DIR / "class_mapping.csv"
mapping_df.to_csv(mapping_path, index=False)

print(mapping_df)
print("\nSaved:", mapping_path)

            class_name  class_index
0            Cardboard            0
1        Food Organics            1
2                Glass            2
3                Metal            3
4  Miscellaneous Trash            4
5                Paper            5
6              Plastic            6
7        Textile Trash            7
8           Vegetation            8

Saved: /content/drive/MyDrive/Deep Learning - RealWaste/02_Splits/class_mapping.csv


In [9]:
import hashlib

def sha256_file(path):
    sha256 = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            sha256.update(chunk)

    return sha256.hexdigest()

split_hash = sha256_file(split_path)
mapping_hash = sha256_file(mapping_path)

print("split.csv SHA256:")
print(split_hash)

print("\nclass_mapping.csv SHA256:")
print(mapping_hash)

split.csv SHA256:
043e68e8028e402f2c2d04cdfa48e44b54c9ac6099981387f2859a0724aa4b1b

class_mapping.csv SHA256:
a99922d406b2a308ff6e85e0015dc675ad24be07f7e481ff4f0a1ed65a8aed8d


In [10]:
manifest_path = SPLIT_DIR / "split_manifest.txt"

with open(manifest_path, "w") as f:
    f.write("RealWaste Fixed Split\n")
    f.write("=====================\n")
    f.write("Seed: 42\n")
    f.write("Train: 3326\n")
    f.write("Validation: 713\n")
    f.write("Test: 713\n")
    f.write("Total: 4752\n")
    f.write(f"split.csv SHA256: {split_hash}\n")
    f.write(f"class_mapping.csv SHA256: {mapping_hash}\n")

print("Saved:", manifest_path)

Saved: /content/drive/MyDrive/Deep Learning - RealWaste/02_Splits/split_manifest.txt


After generating the split:
1. Save the class mapping.
2. Verify every class exists in all three splits.
3. Never regenerate this split independently for another model.
4. Commit/copy the split file as a fixed project artifact.
